# Data Transformation and Relational Logic with Pandas

This notebook covers:
- Groupby and aggregation
- Split-Apply-Combine pattern
- Multiple aggregations
- Merging and joining DataFrames
- Inner, left, right, and outer joins
- Concatenating DataFrames vertically and horizontally
- Pivot tables
- Melting / unpivoting data
- Data engineering examples
- Interview questions


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)


pandas version: 2.2.2


## 2. Create Sample E-Commerce Datasets

The examples use small datasets that represent common data engineering tables:
- orders
- customers
- products
- order items


In [ ]:
orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    "customer_id": [501, 502, 503, 501, 504, 505, 999],
    "order_date": [
        "2026-01-01", "2026-01-01", "2026-01-02",
        "2026-01-02", "2026-01-03", "2026-01-03", "2026-01-04"
    ],
    "city": ["Delhi", "Mumbai", "Bangalore", "Delhi", "Pune", "Chennai", "Hyderabad"],
    "status": ["completed", "completed", "pending", "completed", "cancelled", "completed", "completed"],
    "amount": [250.50, 900.00, 500.00, 1200.00, 0.00, 750.00, 300.00]
})

customers = pd.DataFrame({
    "customer_id": [501, 502, 503, 504, 505, 506],
    "customer_name": ["Riya", "Aarav", "Kabir", "Meera", "Isha", "Rahul"],
    "segment": ["Premium", "Standard", "Standard", "Premium", "Standard", "Premium"],
    "signup_city": ["Delhi", "Mumbai", "Bangalore", "Pune", "Chennai", "Jaipur"]
})

products = pd.DataFrame({
    "product_id": [201, 202, 203, 204, 205],
    "product_name": ["Laptop", "Phone", "Shoes", "Keyboard", "Monitor"],
    "category": ["Electronics", "Electronics", "Fashion", "Electronics", "Electronics"],
    "price": [80000, 40000, 3000, 1500, 12000]
})

order_items = pd.DataFrame({
    "order_id": [1001, 1001, 1002, 1003, 1004, 1004, 1006, 1007],
    "product_id": [202, 204, 201, 203, 201, 205, 203, 999],
    "quantity": [1, 2, 1, 3, 1, 1, 2, 1]
})

print("Orders")
print(orders)

print("Customers")
print(customers)

print("Products")
print(products)

print("Order Items")
print(order_items)


Orders
   order_id  customer_id  order_date       city     status  amount
0      1001          501  2026-01-01      Delhi  completed   250.5
1      1002          502  2026-01-01     Mumbai  completed   900.0
2      1003          503  2026-01-02  Bangalore    pending   500.0
3      1004          501  2026-01-02      Delhi  completed  1200.0
4      1005          504  2026-01-03       Pune  cancelled     0.0
5      1006          505  2026-01-03    Chennai  completed   750.0
6      1007          999  2026-01-04  Hyderabad  completed   300.0
Customers
   customer_id customer_name   segment signup_city
0          501          Riya   Premium       Delhi
1          502         Aarav  Standard      Mumbai
2          503         Kabir  Standard   Bangalore
3          504         Meera   Premium        Pune
4          505          Isha  Standard     Chennai
5          506         Rahul   Premium      Jaipur
Products
   product_id product_name     category  price
0         201       Laptop  Electr

## 3. Groupby: Basic Aggregation

Groupby is used to split data into groups and calculate summary metrics.


In [ ]:
city_revenue = orders.groupby("city")["amount"].sum()

print(city_revenue)
print(type(city_revenue))


city
Bangalore     500.0
Chennai       750.0
Delhi        1450.5
Hyderabad     300.0
Mumbai        900.0
Pune            0.0
Name: amount, dtype: float64
<class 'pandas.core.series.Series'>


## 4. Groupby Result as a DataFrame


In [ ]:
city_revenue_df = orders.groupby("city", as_index=False)["amount"].sum()

print(city_revenue_df)
print(type(city_revenue_df))


        city  amount
0  Bangalore   500.0
1    Chennai   750.0
2      Delhi  1450.5
3  Hyderabad   300.0
4     Mumbai   900.0
5       Pune     0.0
<class 'pandas.core.frame.DataFrame'>


## 5. Split-Apply-Combine Pattern

Groupby follows the Split-Apply-Combine pattern:

1. Split data into groups
2. Apply aggregation or transformation
3. Combine results into a new output


In [ ]:
# Split by status
# Apply sum on amount
# Combine into one result

status_revenue = orders.groupby("status", as_index=False)["amount"].sum()

print(status_revenue)


      status  amount
0  cancelled     0.0
1  completed  3400.5
2    pending   500.0


## 6. Multiple Aggregations on One Column


In [ ]:
city_amount_summary = orders.groupby("city")["amount"].agg(["min", "max", "mean", "sum", "count"])

print(city_amount_summary)


             min     max    mean     sum  count
city                                           
Bangalore  500.0   500.0  500.00   500.0      1
Chennai    750.0   750.0  750.00   750.0      1
Delhi      250.5  1200.0  725.25  1450.5      2
Hyderabad  300.0   300.0  300.00   300.0      1
Mumbai     900.0   900.0  900.00   900.0      1
Pune         0.0     0.0    0.00     0.0      1


## 7. Multiple Aggregations with Named Columns


In [ ]:
city_summary = orders.groupby("city", as_index=False).agg(
    min_amount=("amount", "min"),
    max_amount=("amount", "max"),
    avg_amount=("amount", "mean"),
    total_amount=("amount", "sum"),
    order_count=("order_id", "count")
)

print(city_summary)


        city  min_amount  max_amount  avg_amount  total_amount  order_count
0  Bangalore       500.0       500.0      500.00         500.0            1
1    Chennai       750.0       750.0      750.00         750.0            1
2      Delhi       250.5      1200.0      725.25        1450.5            2
3  Hyderabad       300.0       300.0      300.00         300.0            1
4     Mumbai       900.0       900.0      900.00         900.0            1
5       Pune         0.0         0.0        0.00           0.0            1


## 8. Groupby with Multiple Columns


In [ ]:
city_status_summary = orders.groupby(["city", "status"], as_index=False).agg(
    total_amount=("amount", "sum"),
    order_count=("order_id", "count")
)

print(city_status_summary)


        city     status  total_amount  order_count
0  Bangalore    pending         500.0            1
1    Chennai  completed         750.0            1
2      Delhi  completed        1450.5            2
3  Hyderabad  completed         300.0            1
4     Mumbai  completed         900.0            1
5       Pune  cancelled           0.0            1


## 9. Data Engineering Use Case: Daily Revenue Summary


In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

orders["order_year"] = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month
orders["order_day"] = orders["order_date"].dt.day

daily_revenue = orders.groupby("order_date", as_index=False).agg(
    total_revenue=("amount", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("amount", "mean")
)

print(daily_revenue)


  order_date  total_revenue  order_count  avg_order_value
0 2026-01-01         1150.5            2           575.25
1 2026-01-02         1700.0            2           850.00
2 2026-01-03          750.0            2           375.00
3 2026-01-04          300.0            1           300.00


## 10. Filter Before Groupby

Data pipelines often filter clean records before aggregation.


In [ ]:
completed_orders = orders[orders["status"] == "completed"]

completed_city_summary = completed_orders.groupby("city", as_index=False).agg(
    completed_revenue=("amount", "sum"),
    completed_order_count=("order_id", "count")
)

print(completed_city_summary)


        city  completed_revenue  completed_order_count
0    Chennai              750.0                      1
1      Delhi             1450.5                      2
2  Hyderabad              300.0                      1
3     Mumbai              900.0                      1


# Merging and Joining


## 11. Why Merge DataFrames?

In data engineering, data usually comes from multiple tables.

Examples:
- orders table has customer_id
- customers table has customer details
- order_items table has product_id
- products table has product details

Merging combines related datasets using keys.


## 12. Inner Join

An inner join keeps only matching records from both DataFrames.


In [ ]:
orders_customers_inner = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="inner"
)

print(orders_customers_inner)


   order_id  customer_id order_date       city     status  amount  order_year  \
0      1001          501 2026-01-01      Delhi  completed   250.5        2026   
1      1002          502 2026-01-01     Mumbai  completed   900.0        2026   
2      1003          503 2026-01-02  Bangalore    pending   500.0        2026   
3      1004          501 2026-01-02      Delhi  completed  1200.0        2026   
4      1005          504 2026-01-03       Pune  cancelled     0.0        2026   
5      1006          505 2026-01-03    Chennai  completed   750.0        2026   

   order_month  order_day customer_name   segment signup_city  
0            1          1          Riya   Premium       Delhi  
1            1          1         Aarav  Standard      Mumbai  
2            1          2         Kabir  Standard   Bangalore  
3            1          2          Riya   Premium       Delhi  
4            1          3         Meera   Premium        Pune  
5            1          3          Isha  Standar

## 13. Left Join

A left join keeps all records from the left DataFrame and matching records from the right DataFrame.

If there is no match, right-side columns become null.


In [ ]:
orders_customers_left = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left"
)

print(orders_customers_left)


   order_id  customer_id order_date       city     status  amount  order_year  \
0      1001          501 2026-01-01      Delhi  completed   250.5        2026   
1      1002          502 2026-01-01     Mumbai  completed   900.0        2026   
2      1003          503 2026-01-02  Bangalore    pending   500.0        2026   
3      1004          501 2026-01-02      Delhi  completed  1200.0        2026   
4      1005          504 2026-01-03       Pune  cancelled     0.0        2026   
5      1006          505 2026-01-03    Chennai  completed   750.0        2026   
6      1007          999 2026-01-04  Hyderabad  completed   300.0        2026   

   order_month  order_day customer_name   segment signup_city  
0            1          1          Riya   Premium       Delhi  
1            1          1         Aarav  Standard      Mumbai  
2            1          2         Kabir  Standard   Bangalore  
3            1          2          Riya   Premium       Delhi  
4            1          3      

## 14. Right Join

A right join keeps all records from the right DataFrame and matching records from the left DataFrame.


In [ ]:
orders_customers_right = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="right"
)

print(orders_customers_right)


   order_id  customer_id order_date       city     status  amount  order_year  \
0    1001.0          501 2026-01-01      Delhi  completed   250.5      2026.0   
1    1004.0          501 2026-01-02      Delhi  completed  1200.0      2026.0   
2    1002.0          502 2026-01-01     Mumbai  completed   900.0      2026.0   
3    1003.0          503 2026-01-02  Bangalore    pending   500.0      2026.0   
4    1005.0          504 2026-01-03       Pune  cancelled     0.0      2026.0   
5    1006.0          505 2026-01-03    Chennai  completed   750.0      2026.0   
6       NaN          506        NaT        NaN        NaN     NaN         NaN   

   order_month  order_day customer_name   segment signup_city  
0          1.0        1.0          Riya   Premium       Delhi  
1          1.0        2.0          Riya   Premium       Delhi  
2          1.0        1.0         Aarav  Standard      Mumbai  
3          1.0        2.0         Kabir  Standard   Bangalore  
4          1.0        3.0      

## 15. Outer Join

An outer join keeps all records from both DataFrames.

Rows without matches will contain null values.


In [ ]:
orders_customers_outer = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="outer"
)

print(orders_customers_outer)


   order_id  customer_id order_date       city     status  amount  order_year  \
0    1001.0          501 2026-01-01      Delhi  completed   250.5      2026.0   
1    1004.0          501 2026-01-02      Delhi  completed  1200.0      2026.0   
2    1002.0          502 2026-01-01     Mumbai  completed   900.0      2026.0   
3    1003.0          503 2026-01-02  Bangalore    pending   500.0      2026.0   
4    1005.0          504 2026-01-03       Pune  cancelled     0.0      2026.0   
5    1006.0          505 2026-01-03    Chennai  completed   750.0      2026.0   
6       NaN          506        NaT        NaN        NaN     NaN         NaN   
7    1007.0          999 2026-01-04  Hyderabad  completed   300.0      2026.0   

   order_month  order_day customer_name   segment signup_city  
0          1.0        1.0          Riya   Premium       Delhi  
1          1.0        2.0          Riya   Premium       Delhi  
2          1.0        1.0         Aarav  Standard      Mumbai  
3          1.0

## 16. Join Indicator

The indicator column shows where each row came from:
- left_only
- right_only
- both


In [ ]:
orders_customers_audit = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="outer",
    indicator=True
)

print(orders_customers_audit)


   order_id  customer_id order_date       city     status  amount  order_year  \
0    1001.0          501 2026-01-01      Delhi  completed   250.5      2026.0   
1    1004.0          501 2026-01-02      Delhi  completed  1200.0      2026.0   
2    1002.0          502 2026-01-01     Mumbai  completed   900.0      2026.0   
3    1003.0          503 2026-01-02  Bangalore    pending   500.0      2026.0   
4    1005.0          504 2026-01-03       Pune  cancelled     0.0      2026.0   
5    1006.0          505 2026-01-03    Chennai  completed   750.0      2026.0   
6       NaN          506        NaT        NaN        NaN     NaN         NaN   
7    1007.0          999 2026-01-04  Hyderabad  completed   300.0      2026.0   

   order_month  order_day customer_name   segment signup_city      _merge  
0          1.0        1.0          Riya   Premium       Delhi        both  
1          1.0        2.0          Riya   Premium       Delhi        both  
2          1.0        1.0         Aarav  S

## 17. Data Quality Use Case: Find Orders with Missing Customer Records


In [ ]:
orders_with_customer_check = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left",
    indicator=True
)

missing_customer_orders = orders_with_customer_check[
    orders_with_customer_check["_merge"] == "left_only"
]

print(missing_customer_orders)


   order_id  customer_id order_date       city     status  amount  order_year  \
6      1007          999 2026-01-04  Hyderabad  completed   300.0        2026   

   order_month  order_day customer_name segment signup_city     _merge  
6            1          4           NaN     NaN         NaN  left_only  


## 18. Merge on Different Column Names


In [ ]:
customer_reference = customers.rename(columns={"customer_id": "id"})

merged_different_names = pd.merge(
    orders,
    customer_reference,
    left_on="customer_id",
    right_on="id",
    how="left"
)

print(merged_different_names)


   order_id  customer_id order_date       city     status  amount  order_year  \
0      1001          501 2026-01-01      Delhi  completed   250.5        2026   
1      1002          502 2026-01-01     Mumbai  completed   900.0        2026   
2      1003          503 2026-01-02  Bangalore    pending   500.0        2026   
3      1004          501 2026-01-02      Delhi  completed  1200.0        2026   
4      1005          504 2026-01-03       Pune  cancelled     0.0        2026   
5      1006          505 2026-01-03    Chennai  completed   750.0        2026   
6      1007          999 2026-01-04  Hyderabad  completed   300.0        2026   

   order_month  order_day     id customer_name   segment signup_city  
0            1          1  501.0          Riya   Premium       Delhi  
1            1          1  502.0         Aarav  Standard      Mumbai  
2            1          2  503.0         Kabir  Standard   Bangalore  
3            1          2  501.0          Riya   Premium       Delh

## 19. Multi-Step Join: Orders, Order Items, Products


In [ ]:
orders_with_items = pd.merge(
    orders,
    order_items,
    on="order_id",
    how="left"
)

full_order_details = pd.merge(
    orders_with_items,
    products,
    on="product_id",
    how="left"
)

print(full_order_details)


   order_id  customer_id order_date       city     status  amount  order_year  \
0      1001          501 2026-01-01      Delhi  completed   250.5        2026   
1      1001          501 2026-01-01      Delhi  completed   250.5        2026   
2      1002          502 2026-01-01     Mumbai  completed   900.0        2026   
3      1003          503 2026-01-02  Bangalore    pending   500.0        2026   
4      1004          501 2026-01-02      Delhi  completed  1200.0        2026   
5      1004          501 2026-01-02      Delhi  completed  1200.0        2026   
6      1005          504 2026-01-03       Pune  cancelled     0.0        2026   
7      1006          505 2026-01-03    Chennai  completed   750.0        2026   
8      1007          999 2026-01-04  Hyderabad  completed   300.0        2026   

   order_month  order_day  product_id  quantity product_name     category  \
0            1          1       202.0       1.0        Phone  Electronics   
1            1          1       204

## 20. Add Derived Columns After Merge


In [ ]:
full_order_details["line_total"] = full_order_details["quantity"] * full_order_details["price"]

print(full_order_details[["order_id", "product_id", "product_name", "quantity", "price", "line_total"]])


   order_id  product_id product_name  quantity    price  line_total
0      1001       202.0        Phone       1.0  40000.0     40000.0
1      1001       204.0     Keyboard       2.0   1500.0      3000.0
2      1002       201.0       Laptop       1.0  80000.0     80000.0
3      1003       203.0        Shoes       3.0   3000.0      9000.0
4      1004       201.0       Laptop       1.0  80000.0     80000.0
5      1004       205.0      Monitor       1.0  12000.0     12000.0
6      1005         NaN          NaN       NaN      NaN         NaN
7      1006       203.0        Shoes       2.0   3000.0      6000.0
8      1007       999.0          NaN       1.0      NaN         NaN


## 21. Category-Level Revenue from Joined Data


In [ ]:
category_revenue = full_order_details.groupby("category", as_index=False).agg(
    total_line_revenue=("line_total", "sum"),
    total_quantity=("quantity", "sum"),
    item_count=("product_id", "count")
)

print(category_revenue)


      category  total_line_revenue  total_quantity  item_count
0  Electronics            215000.0             6.0           5
1      Fashion             15000.0             5.0           2


# Concatenating DataFrames


## 22. Vertical Concatenation

Vertical concatenation stacks rows.

This is useful when combining daily batches with the same schema.


In [ ]:
orders_day_1 = pd.DataFrame({
    "order_id": [2001, 2002],
    "customer_id": [601, 602],
    "amount": [300, 450]
})

orders_day_2 = pd.DataFrame({
    "order_id": [2003, 2004],
    "customer_id": [603, 604],
    "amount": [700, 150]
})

all_orders = pd.concat([orders_day_1, orders_day_2], axis=0, ignore_index=True)

print(all_orders)


   order_id  customer_id  amount
0      2001          601     300
1      2002          602     450
2      2003          603     700
3      2004          604     150


## 23. Horizontal Concatenation

Horizontal concatenation adds columns side by side.

This is useful when two DataFrames have aligned rows.


In [ ]:
customer_base = pd.DataFrame({
    "customer_id": [501, 502, 503],
    "customer_name": ["Riya", "Aarav", "Kabir"]
})

customer_metrics = pd.DataFrame({
    "total_orders": [5, 2, 3],
    "total_spend": [4000, 1200, 2500]
})

customer_profile = pd.concat([customer_base, customer_metrics], axis=1)

print(customer_profile)


   customer_id customer_name  total_orders  total_spend
0          501          Riya             5         4000
1          502         Aarav             2         1200
2          503         Kabir             3         2500


## 24. Concat with Mismatched Columns


In [ ]:
batch_a = pd.DataFrame({
    "order_id": [3001, 3002],
    "amount": [500, 600]
})

batch_b = pd.DataFrame({
    "order_id": [3003, 3004],
    "amount": [700, 800],
    "status": ["completed", "pending"]
})

combined_batches = pd.concat([batch_a, batch_b], axis=0, ignore_index=True)

print(combined_batches)


   order_id  amount     status
0      3001     500        NaN
1      3002     600        NaN
2      3003     700  completed
3      3004     800    pending


# Pivoting and Reshaping


## 25. Pivot Table

A pivot table creates a summary report from detailed rows.


In [ ]:
pivot_city_status = pd.pivot_table(
    orders,
    values="amount",
    index="city",
    columns="status",
    aggfunc="sum",
    fill_value=0
)

print(pivot_city_status)


status     cancelled  completed  pending
city                                    
Bangalore        0.0        0.0    500.0
Chennai          0.0      750.0      0.0
Delhi            0.0     1450.5      0.0
Hyderabad        0.0      300.0      0.0
Mumbai           0.0      900.0      0.0
Pune             0.0        0.0      0.0


## 26. Pivot Table with Multiple Aggregations


In [ ]:
pivot_city_summary = pd.pivot_table(
    orders,
    values="amount",
    index="city",
    columns="status",
    aggfunc=["sum", "count"],
    fill_value=0
)

print(pivot_city_summary)


                sum                       count                  
status    cancelled completed pending cancelled completed pending
city                                                             
Bangalore       0.0       0.0   500.0         0         0       1
Chennai         0.0     750.0     0.0         0         1       0
Delhi           0.0    1450.5     0.0         0         2       0
Hyderabad       0.0     300.0     0.0         0         1       0
Mumbai          0.0     900.0     0.0         0         1       0
Pune            0.0       0.0     0.0         1         0       0


## 27. Data Engineering Use Case: Daily Revenue Report


In [ ]:
daily_status_report = pd.pivot_table(
    orders,
    values="amount",
    index="order_date",
    columns="status",
    aggfunc="sum",
    fill_value=0
)

print(daily_status_report)


status      cancelled  completed  pending
order_date                               
2026-01-01        0.0     1150.5      0.0
2026-01-02        0.0     1200.0    500.0
2026-01-03        0.0      750.0      0.0
2026-01-04        0.0      300.0      0.0


## 28. Melt: Wide Format to Long Format

Melt converts columns into rows.

This is useful when preparing wide reporting data for analytics pipelines.


In [ ]:
wide_sales = pd.DataFrame({
    "store_id": [1, 2, 3],
    "jan_sales": [1000, 1200, 900],
    "feb_sales": [1100, 1300, 950],
    "mar_sales": [1050, 1250, 1000]
})

print(wide_sales)


   store_id  jan_sales  feb_sales  mar_sales
0         1       1000       1100       1050
1         2       1200       1300       1250
2         3        900        950       1000


In [ ]:
long_sales = pd.melt(
    wide_sales,
    id_vars=["store_id"],
    value_vars=["jan_sales", "feb_sales", "mar_sales"],
    var_name="month",
    value_name="sales"
)

print(long_sales)


   store_id      month  sales
0         1  jan_sales   1000
1         2  jan_sales   1200
2         3  jan_sales    900
3         1  feb_sales   1100
4         2  feb_sales   1300
5         3  feb_sales    950
6         1  mar_sales   1050
7         2  mar_sales   1250
8         3  mar_sales   1000


## 29. Clean Month Names After Melt


In [ ]:
long_sales["month"] = long_sales["month"].str.replace("_sales", "", regex=False)

print(long_sales)


   store_id month  sales
0         1   jan   1000
1         2   jan   1200
2         3   jan    900
3         1   feb   1100
4         2   feb   1300
5         3   feb    950
6         1   mar   1050
7         2   mar   1250
8         3   mar   1000


## 30. Pivot Long Data Back to Wide Format


In [ ]:
wide_again = long_sales.pivot_table(
    values="sales",
    index="store_id",
    columns="month",
    aggfunc="sum"
).reset_index()

print(wide_again)


month  store_id   feb   jan   mar
0             1  1100  1000  1050
1             2  1300  1200  1250
2             3   950   900  1000


# End-to-End Data Transformation Example


## 31. Business Problem: Build Customer Revenue Summary

Steps:
1. Join orders with customers
2. Keep completed orders
3. Group by customer segment
4. Calculate revenue metrics
5. Create a pivot report by segment and city


In [ ]:
orders_customers = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left"
)

completed_orders_customers = orders_customers[
    orders_customers["status"] == "completed"
]

segment_summary = completed_orders_customers.groupby("segment", as_index=False).agg(
    total_revenue=("amount", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("amount", "mean")
)

print(segment_summary)


    segment  total_revenue  order_count  avg_order_value
0   Premium         1450.5            2           725.25
1  Standard         1650.0            2           825.00


In [ ]:
segment_city_pivot = pd.pivot_table(
    completed_orders_customers,
    values="amount",
    index="segment",
    columns="city",
    aggfunc="sum",
    fill_value=0
)

print(segment_city_pivot)


city      Chennai   Delhi  Mumbai
segment                          
Premium       0.0  1450.5     0.0
Standard    750.0     0.0   900.0


## 32. Save Transformed Outputs


In [ ]:
segment_summary.to_csv("segment_summary.csv", index=False)

try:
    segment_summary.to_parquet("segment_summary.parquet", index=False)
    print("CSV and Parquet files created")
except Exception as error:
    print("CSV created. Parquet write failed. Install pyarrow if needed.")
    print(error)


CSV and Parquet files created


# Practice Problems


## 33. Practice Problem 1: Customer-Level Aggregation

Using `orders`, create a customer-level summary with:
- total amount
- minimum order amount
- maximum order amount
- average order amount
- order count


In [ ]:
# Write solution here


## 34. Practice Problem 2: Left Join Orders and Customers

Perform a left join between `orders` and `customers`.

Find orders that do not have matching customer records.


In [ ]:
# Write solution here


## 35. Practice Problem 3: Product Category Revenue

Join `orders`, `order_items`, and `products`.

Create a category-level report with:
- total quantity sold
- total revenue
- number of order lines


In [ ]:
# Write solution here


## 36. Practice Problem 4: Combine Daily Batches

Create three daily order DataFrames and concatenate them vertically.


In [ ]:
# Write solution here


## 37. Practice Problem 5: Pivot Report

Create a pivot table showing total order amount by:
- city as rows
- status as columns


In [ ]:
# Write solution here


## 38. Practice Problem 6: Melt Sales Data

Create a wide sales DataFrame with monthly columns.

Convert it to long format using `pd.melt()`.


In [ ]:
# Write solution here


# Interview Questions


## 39. What is the difference between `merge()` and `concat()` in Pandas?

`merge()` combines DataFrames using matching key columns, similar to SQL joins.

Example:

```python
pd.merge(orders, customers, on="customer_id", how="left")
```

`concat()` stacks or attaches DataFrames along an axis.

Vertical concat adds rows:

```python
pd.concat([batch_1, batch_2], axis=0)
```

Horizontal concat adds columns:

```python
pd.concat([df_1, df_2], axis=1)
```


## 40. Explain the Split-Apply-Combine Strategy in Groupby

Split-Apply-Combine means:

1. Split the DataFrame into groups based on one or more columns.
2. Apply an operation to each group, such as sum, mean, min, max, or count.
3. Combine the results into a final DataFrame or Series.

Example:

```python
orders.groupby("city")["amount"].sum()
```


## 41. How Do You Perform a Left Join Between Two DataFrames?

Use `pd.merge()` with `how="left"`.

Example:

```python
result = pd.merge(
    orders,
    customers,
    on="customer_id",
    how="left"
)
```

This keeps all rows from `orders` and adds matching customer information where available.


## 42. What Is an Inner Join?

An inner join returns only matching records from both DataFrames.

Rows without a matching key in either DataFrame are excluded.


## 43. What Is an Outer Join?

An outer join returns all records from both DataFrames.

If a record does not have a match, missing columns are filled with null values.


## 44. What Is the Difference Between Pivot and Melt?

Pivot creates a wider summary table.

Melt converts wide data into long data.

Pivot is useful for reports.

Melt is useful for normalization and analytics-friendly formats.


## 45. Summary

Key takeaways:
- `groupby()` is used for grouped aggregation.
- Split-Apply-Combine is the mental model behind groupby.
- `merge()` performs SQL-like joins.
- `concat()` combines DataFrames by rows or columns.
- `pivot_table()` creates summary reports.
- `melt()` converts wide data into long format.
- These operations are essential for transforming raw datasets into analytics-ready tables.
